This notebook contains the instructions for creating count tables for the Tomo-seq data from the mapping results.

The conda environment (kernel) for this notebook:
```
conda create -n count_tables
conda activate count_tables
conda install jupyter_client r-base r-irkernel r-tidyverse bioconductor-tximport bioconductor-rtracklayer

```

## Dependencies

In [ ]:
library(dplyr)
library(stringr)
library(fs)

# --- Import Utilities ---
r_utils_path <- path("utils", "r_utils")
source(path(r_utils_path, "quantification_utils.r"))
source(path(r_utils_path, "barcode_processing_utils.r"))

## Salmon: gene-level counts from mapping results

### GTF annotation files

In [2]:
acomys_gtf_url <- 'https://ftp.ensembl.org/pub/rapid-release/species/Acomys_cahirinus/GCA_029890205.1/ensembl/geneset/2023_11/Acomys_cahirinus-GCA_029890205.1-2023_11-genes.gtf.gz'
acomys_gtf <- rtracklayer::import(acomys_gtf_url)
acomys_gtf <- as_tibble(acomys_gtf)

### Format the Salmon mapping results
Create a directory per sample with the Salmon mapping results for every barcode.  

In [ ]:
mapped_dir = path('datasets', 'Tomoseq_mapped')
if (!is_dir(mapped_dir)) {
    error("Mapped directory not found.")
}

# --- Iterate over sample directories ---
# ---------------------------------------
sample_mapping_dir_list = dir_ls(mapped_dir, type = "directory")
if (length(sample_mapping_dir_list) == 0) {
    error("No sample directories found.")
} else {
    message(paste("Found", length(sample_mapping_dir_list), "sample directories."))
}
for (sample_mapping_dir in sample_mapping_dir_list) {
    message(paste("Processing sample", sample_mapping_dir))
    # Create a Salmon result directory for the sample
    sample_id <- basename(sample_mapping_dir)
    sample_id <- str_split_1(sample_id, "_")[1]
    salmon_result_dir = path(sample_mapping_dir, paste0("Salmon_mapping_results_", sample_id))
    dir_create(salmon_result_dir)
    
    # --- Iterate over barcode directories ---
    # ----------------------------------------
    barcode_mapping_dir_list = dir_ls(sample_mapping_dir, type = "directory", glob = "*bcode_*")
    if (length(barcode_mapping_dir_list) == 0) {
        message("No barcode directories found.")
        next
    } else {
        message(paste("Found", length(barcode_mapping_dir_list), "barcode directories."))
    }
    for (barcode_mapping_dir in barcode_mapping_dir_list) {
        message(paste("Processing", barcode_mapping_dir))
        
        # Extract barcode sequence and ID from the directory name
        barcode_info <- extract_barcode_info(barcode_mapping_dir)
        if (is.null(barcode_info$barcode)) {
            message(paste("No barcode found in", barcode_mapping_dir))
        } else {
            barcode <- barcode_info$barcode
            barcode_id <- barcode_info$barcode_id
            message(paste("Barcode:", barcode_info$barcode, "barcode ID:", barcode_info$barcode_id))
        }

        # Fetch the Salmon quant.sf file
        salmon_quant_file <- dir_ls(barcode_mapping_dir, regexp = "quant.sf", recurse = TRUE)
        if (length(salmon_quant_file) == 0) {
            message(paste("No Salmon quant.sf file found in", barcode_mapping_dir))
            next
        } else {
            message(paste("Salmon quant.sf file found:", salmon_quant_file))
        }
        new_salmon_quant_file_name = paste0('bcode_', barcode_id, '_', barcode, '_Salmon_quant.sf')
        file_copy(salmon_quant_file, path(salmon_result_dir, new_salmon_quant_file_name))
    }
}


### Create count tables

In [11]:
mapped_dir = path('datasets', 'Tomoseq_mapped')
if (!is_dir(mapped_dir)) {
    error("Mapped directory not found.")
}

gene_counts_dir = path('datasets', 'Tomoseq_gene_counts')
if (!is_dir(gene_counts_dir)) {
    dir_create(gene_counts_dir)
}

# --- Iterate over sample directories ---
# ---------------------------------------
sample_mapping_dir_list = dir_ls(mapped_dir, type = "directory")
if (length(sample_mapping_dir_list) == 0) {
    error("No sample directories found.")
} else {
    message(paste("Found", length(sample_mapping_dir_list), "sample directories."))
}
for (sample_mapping_dir in sample_mapping_dir_list) {
    sample_id <- basename(sample_mapping_dir)
    sample_id <- str_split_1(sample_id, "_")[1]
    message(paste("Processing sample", sample_id))

    # Fetch the Salmon results directory
    sample_salmon_result_dir <- dir_ls(sample_mapping_dir, regexp = "Salmon_mapping_results", recurse = TRUE, type = "directory")
    # Check that there is only one Salmon results directory
    if (length(sample_salmon_result_dir) == 0) {
        message(paste("No Salmon results directory found in", sample_mapping_dir, "- Skipping sample."))
        next
    } 
    if (length(sample_salmon_result_dir) > 1) {
        stop(paste("Found", length(sample_salmon_result_dir), "Salmon results directories in", sample_mapping_dir))
    }
    # Check that the Salmon results directory is not empty
    if (length(dir_ls(sample_salmon_result_dir)) == 0) {
        message(paste("Salmon results directory is empty:", sample_salmon_result_dir, "- Skipping sample."))
        next
    }
    message(paste("Salmon results directory:", sample_salmon_result_dir))
    sample_txi <- salmon_summarize_gene_level(gtf_df = acomys_gtf, salmon_results_dir = sample_salmon_result_dir)
    sample_counts <- sample_txi$counts

    # --- Reorder the columns by the barcode ID number ---
    numeric_part <- str_extract(colnames(sample_counts), "(?<=^bcode_)\\d+")
    numeric_part <- as.numeric(numeric_part)
    if (any(is.na(numeric_part))) {
    warning("Could not extract a valid number from all column names.")
    }
    sort_order_indices <- order(numeric_part)
    sample_counts <- sample_counts[, sort_order_indices]
    
    counts_output_file <- path(gene_counts_dir, paste0(sample_id, '_gene_counts_salmon.rds'))
    saveRDS(sample_counts, counts_output_file)
}


Found 12 sample directories.



Processing sample AC-MI14D-01-RPI4

Salmon results directory: datasets/Tomoseq_mapped/AC-MI14D-01-RPI4_mapped/Salmon_mapping_resultsAC-MI14D-01-RPI4

Found 96 Salmon .sf files in directory: datasets/Tomoseq_mapped/AC-MI14D-01-RPI4_mapped/Salmon_mapping_resultsAC-MI14D-01-RPI4

reading in files with read_tsv

1 
2 
3 
4 
5 
6 
7 
8 
9 
10 
11 
12 
13 
14 
15 
16 
17 
18 
19 
20 
21 
22 
23 
24 
25 
26 
27 
28 
29 
30 
31 
32 
33 
34 
35 
36 
37 
38 
39 
40 
41 
42 
43 
44 
45 
46 
47 
48 
49 
50 
51 
52 
53 
54 
55 
56 
57 
58 
59 
60 
61 
62 
63 
64 
65 
66 
67 
68 
69 
70 
71 
72 
73 
74 
75 
76 
77 
78 
79 
80 
81 
82 
83 
84 
85 
86 
87 
88 
89 
90 
91 
92 
93 
94 
95 
96 


summarizing abundance

summarizing counts

summarizing length

Processing sample AC-MI14D-02-RPI5

Salmon results directory: datasets/Tomoseq_mapped/AC-MI14D-02-RPI5_mapped/Salmon_mapping_resultsAC-MI14D-02-RPI5

Found 96 Salmon .sf files in directory: datasets/Tomoseq_mapped/AC-MI14D-02-RPI5_mapped/Salmon_mappi